# Simplest BERT-style Transformer on BabiStories
Encoder-only masked language model. Compact and based on the previous notebook style.

In [ ]:
import torch,random,math,re,json
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from collections import Counter
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cfg={
    "d_model":64,
    "d_ff":128,
    "num_heads":4,
    "num_layers":2,
    "max_vocab":8000,
    "seq_len":64,
    "batch_size":4,
    "lr":3e-4,
    "epochs":20,
    "mask_prob":0.15,
    "use_wq":False,
    "use_aq":False,
    "w_bits":8,
    "a_bits":8,
    "use_ws":False,
    "ws_ratio":0.0,
    "attn_top_k":None,
    "norm":"post"
}

## Data

##### Load BabiStories Dataset

In [ ]:
def read_texts_from_file(p):
    texts=[]
    if p.suffix.lower()==".txt":
        x=p.read_text(encoding="utf-8",errors="ignore")
        for t in re.split(r"\n\s*\n|\n",x):
            t=" ".join(t.split())
            if len(t.split())>20:texts.append(t)
    elif p.suffix.lower()==".jsonl":
        for line in p.open(encoding="utf-8",errors="ignore"):
            if line.strip():
                d=json.loads(line);t=d.get("text") or d.get("story") or d.get("content") or ""
                t=" ".join(t.split())
                if len(t.split())>20:texts.append(t)
    elif p.suffix.lower()==".json":
        d=json.loads(p.read_text(encoding="utf-8",errors="ignore"))
        if isinstance(d,list):
            for e in d:
                t=e.get("text") or e.get("story") or e.get("content") or ""
                t=" ".join(t.split())
                if len(t.split())>20:texts.append(t)
    return texts

def load_babistories(folder="BabiStories/data/extracted"):
    folder=Path(folder);texts=[]
    if not folder.exists():raise FileNotFoundError(folder)
    for p in folder.rglob("*"):
        if p.suffix.lower() in [".txt",".jsonl",".json"]:texts+=read_texts_from_file(p)
    if len(texts)==0:raise ValueError("No .txt/.json/.jsonl stories found")
    return texts

texts=load_babistories()
random.seed(42);random.shuffle(texts)
print("texts:",len(texts))
print(texts[0][:300])

In [ ]:
def build_context(texts,cfg):
    sp=["<PAD>","<MASK>","<UNK>"];c=Counter()
    for t in texts:c.update(t.split())
    vocab=sp+[w for w,_ in c.most_common(cfg["max_vocab"]-len(sp))]
    stoi={w:i for i,w in enumerate(vocab)};itos={i:w for w,i in stoi.items()}
    return {"stoi":stoi,"itos":itos,"vocab":vocab,"pad":stoi["<PAD>"],"mask":stoi["<MASK>"],"unk":stoi["<UNK>"],"seq_len":cfg["seq_len"]}

def encode(text,ctx,max_len):
    ids=[ctx["stoi"].get(w,ctx["unk"]) for w in text.split()]
    ids=ids[:max_len];ids=ids+[ctx["pad"]]*(max_len-len(ids))
    return ids

def make_examples(texts,ctx,max_examples=10000):
    ex=[]
    for t in texts:
        ids=[ctx["stoi"].get(w,ctx["unk"]) for w in t.split()]
        for i in range(0,max(1,len(ids)-ctx["seq_len"]+1),ctx["seq_len"]):
            s=ids[i:i+ctx["seq_len"]]
            if len(s)>8:ex.append(s+[ctx["pad"]]*(ctx["seq_len"]-len(s)))
            if len(ex)>=max_examples:return ex
    return ex

def make_mlm_batch(data,ctx,batch_size,device):
    b=random.choices(data,k=batch_size) if len(data)<batch_size else random.sample(data,batch_size)
    x=torch.tensor(b,dtype=torch.long,device=device);y=torch.full_like(x,-100);m=x.ne(ctx["pad"])
    r=torch.rand(x.shape,device=device)
    pos=(r<cfg["mask_prob"])&m
    y[pos]=x[pos];x[pos]=ctx["mask"]
    return x,y,x.eq(ctx["pad"])

ctx=build_context(texts,cfg)
examples=make_examples(texts,ctx,max_examples=10000)
random.shuffle(examples)
n=int(0.8*len(examples));v=int(0.1*len(examples))
train_data=examples[:n];valid_data=examples[n:n+v];test_data=examples[n+v:]
print("train:",len(train_data),"valid:",len(valid_data),"test:",len(test_data),"vocab:",len(ctx["vocab"]))

## Quantization and sparsity hooks

In [ ]:
def qste(x,bits,use):
    if not use:return x
    qmax=2**(bits-1)-1;s=x.abs().max().clamp(min=1e-8)/qmax
    y=(x/s).round().clamp(-qmax,qmax)*s
    return x+(y-x).detach()
def sparsify(w,ratio,use):
    if not use or ratio<=0:return w
    th=torch.quantile(w.abs().flatten(),ratio)
    return w*(w.abs()>=th)
def qw(w,cfg):return qste(sparsify(w,cfg["ws_ratio"],cfg["use_ws"]),cfg["w_bits"],cfg["use_wq"])
def qa(x,cfg):return qste(x,cfg["a_bits"],cfg["use_aq"])
def lin(x,l,cfg):return F.linear(qa(x,cfg),qw(l.weight,cfg),l.bias)

## Model

In [ ]:
class MHA(nn.Module):
    def __init__(self,d_model,num_heads,cfg):
        super().__init__();self.h=num_heads;self.dh=d_model//num_heads;self.cfg=cfg
        self.q=nn.Linear(d_model,d_model);self.k=nn.Linear(d_model,d_model);self.v=nn.Linear(d_model,d_model);self.o=nn.Linear(d_model,d_model)
    def forward(self,x,key_pad=None):
        B,T,D=x.shape
        Q=lin(x,self.q,self.cfg).view(B,T,self.h,self.dh).transpose(1,2)
        K=lin(x,self.k,self.cfg).view(B,T,self.h,self.dh).transpose(1,2)
        V=lin(x,self.v,self.cfg).view(B,T,self.h,self.dh).transpose(1,2)
        S=Q@K.transpose(-2,-1)/math.sqrt(self.dh)
        if key_pad is not None:S=S.masked_fill(key_pad[:,None,None,:],-1e9)
        if self.cfg["attn_top_k"] is not None:
            k=min(self.cfg["attn_top_k"],T);th=S.topk(k,dim=-1).values[...,-1,None];S=S.masked_fill(S<th,-1e9)
        A=F.softmax(S,dim=-1);O=(A@V).transpose(1,2).contiguous().view(B,T,D)
        return lin(O,self.o,self.cfg),A

class FFN(nn.Module):
    def __init__(self,d_model,d_ff,cfg):
        super().__init__();self.l1=nn.Linear(d_model,d_ff);self.l2=nn.Linear(d_ff,d_model);self.cfg=cfg
    def forward(self,x):
        return lin(F.relu(lin(x,self.l1,self.cfg)),self.l2,self.cfg)

class Block(nn.Module):
    def __init__(self,d_model,d_ff,num_heads,cfg):
        super().__init__();self.a=MHA(d_model,num_heads,cfg);self.f=FFN(d_model,d_ff,cfg);self.n1=nn.LayerNorm(d_model);self.n2=nn.LayerNorm(d_model);self.cfg=cfg
    def forward(self,x,pad):
        if self.cfg["norm"]=="post":a,A=self.a(x,pad);x=self.n1(x+a);x=self.n2(x+self.f(x))
        else:n=self.n1(x);a,A=self.a(n,pad);x=x+a;x=x+self.f(self.n2(x))
        return x,A

class BERTMLM(nn.Module):
    def __init__(self,vocab_size,cfg,ctx):
        super().__init__();d=cfg["d_model"];self.cfg=cfg;self.ctx=ctx
        self.tok=nn.Embedding(vocab_size,d,padding_idx=ctx["pad"]);self.pos=nn.Embedding(ctx["seq_len"],d)
        self.blocks=nn.ModuleList([Block(d,cfg["d_ff"],cfg["num_heads"],cfg) for _ in range(cfg["num_layers"])])
        self.n=nn.LayerNorm(d);self.out=nn.Linear(d,vocab_size)
    def forward(self,x,pad):
        p=torch.arange(x.size(1),device=x.device)[None,:];h=self.tok(x)+self.pos(p);A=[]
        for b in self.blocks:h,a=b(h,pad);A.append(a)
        return lin(self.n(h),self.out,self.cfg),A

## Train

In [ ]:
model=BERTMLM(len(ctx["vocab"]),cfg,ctx).to(device)
opt=torch.optim.AdamW(model.parameters(),lr=cfg["lr"])

def acc_mlm(logits,y):
    m=y.ne(-100)
    if m.sum().item()==0:return 0.0
    return (logits.argmax(-1).eq(y)&m).sum().item()/m.sum().item()

def step(data,train=True):
    model.train(train);x,y,pad=make_mlm_batch(data,ctx,cfg["batch_size"],device)
    with torch.set_grad_enabled(train):
        logits,A=model(x,pad)
        loss=F.cross_entropy(logits.reshape(-1,logits.size(-1)),y.reshape(-1),ignore_index=-100)
        if train:
            opt.zero_grad();loss.backward();nn.utils.clip_grad_norm_(model.parameters(),1.0);opt.step()
    return loss.item(),acc_mlm(logits,y)

def evaluate(data,n=20):
    loss=0;acc=0
    for _ in range(n):
        l,a=step(data,False);loss+=l;acc+=a
    return loss/n,acc/n

def train_epochs(epochs=20,train_steps=100,valid_steps=20):
    history={"train_loss":[],"train_acc":[],"val_loss":[],"val_acc":[]};best=1e9;state=None
    for e in range(1,epochs+1):
        tl=ta=0
        for _ in range(train_steps):
            l,a=step(train_data,True);tl+=l;ta+=a
        vl,va=evaluate(valid_data,valid_steps);tl/=train_steps;ta/=train_steps
        history["train_loss"].append(tl);history["train_acc"].append(ta);history["val_loss"].append(vl);history["val_acc"].append(va)
        if vl<best:best=vl;state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        print(e,"train_loss",tl,"train_acc",ta,"val_loss",vl,"val_acc",va)
    return state,history

best_state,history=train_epochs(cfg["epochs"],100,20)
model.load_state_dict(best_state)

## Fill mask

In [ ]:
def fill_mask(text,topk=5):
    model.eval()
    ids=encode(text,ctx,ctx["seq_len"])
    x=torch.tensor([ids],dtype=torch.long,device=device);pad=x.eq(ctx["pad"])
    pos=[i for i,v in enumerate(ids) if v==ctx["mask"]]
    if len(pos)==0:return []
    with torch.no_grad():logits,A=model(x,pad)
    out=[]
    for p in pos:
        probs=F.softmax(logits[0,p],dim=-1)
        vals,idx=probs.topk(topk)
        out.append([(ctx["itos"][int(i)],float(v)) for v,i in zip(vals,idx)])
    return out

print(fill_mask("Mary went to the <MASK> .",5))

## Later switches

In [ ]:
# cfg["use_wq"]=True;cfg["use_aq"]=True
# cfg["use_ws"]=True;cfg["ws_ratio"]=0.5
# cfg["attn_top_k"]=8